# Remote Deployment with Hetzner Cloud

This notebook demonstrates deploying a netrun pool server to a cloud VM and running
a network that offloads computation to it.

Steps:
1. Deploy the `app/` folder to a new Hetzner Cloud server
2. Run a network with a remote pool on the provisioned server
3. Tear down the server

**Prerequisites:**
- `hcloud` CLI installed and authenticated (`hcloud context create`)
- An SSH key registered in your Hetzner Cloud project (`hcloud ssh-key list`)
- The corresponding private key available locally
- A `.env` file in this directory (copy `.env.example` and fill in your values)

## Configuration

In [1]:
from pathlib import Path
from dotenv import load_dotenv
import os

load_dotenv()

# --- Loaded from .env (see .env.example) ---
HCLOUD_SSH_KEY_NAME = os.environ["HCLOUD_SSH_KEY_NAME"]
SSH_PRIVATE_KEY_PATH = os.environ["SSH_PRIVATE_KEY_PATH"]

# --- Server settings ---
SERVER_NAME = "netrun-demo"
SERVER_TYPE = "cpx22"          # 2 vCPU, 4 GB RAM
SERVER_IMAGE = "ubuntu-24.04"
SERVER_LOCATION = "fsn1"       # Falkenstein, DE

# --- Deployment settings ---
REMOTE_DIR = "/opt/netrun-app"
POOL_SERVER_PORT = 8765
APP_DIR = str(Path("./app").resolve())

## Deploy

Creates a Hetzner server, uploads the `app/` folder, installs dependencies
with `uv`, starts the pool server, and waits until it's reachable.

In [2]:
from deploy_to_hetzner import deploy_to_hetzner, delete_server

result = deploy_to_hetzner(
    server_name=SERVER_NAME,
    ssh_key_name=HCLOUD_SSH_KEY_NAME,
    ssh_private_key_path=SSH_PRIVATE_KEY_PATH,
    local_folder=APP_DIR,
    remote_dir=REMOTE_DIR,
    net_source="net_config.toml",
    pool_server_port=POOL_SERVER_PORT,
    server_type=SERVER_TYPE,
    server_image=SERVER_IMAGE,
    server_location=SERVER_LOCATION,
    python_version="3.11",
)
print(f"\nServer: {result.server_ip}")
print(f"Pool:   {result.pool_server_url}")

Creating server 'netrun-demo' (cpx22, ubuntu-24.04, fsn1)...
  Server IP: 188.245.171.241
Waiting for SSH...... ready!
Deploying with pyinfra...


No host key for 188.245.171.241 found in known_hosts, accepting & adding to host keys file
Added host key for 188.245.171.241 to known_hosts


Waiting for remote port 8765. ready!
Opening SSH tunnel (localhost:8765 -> 188.245.171.241:8765)...
  Tunnel active (pid 66838)
Pool server ready: ws://localhost:8765

Server: 188.245.171.241
Pool:   ws://localhost:8765


## Run the Network

The `find_primes` function runs on the remote server. We inject a range and
collect the results locally.

In [3]:
import sys

# Add app/ to path so the client can resolve the function factory
sys.path.insert(0, APP_DIR)

from netrun.net import Net
from netrun.net.config._net_config import NetConfig, PoolConfig, RemotePoolConfig
from netrun.net.config._graph import GraphConfig
from netrun.net.config._nodes import NodeConfig, NodeExecutionConfig

client_config = NetConfig(
    pools={
        "remote": PoolConfig(
            spec=RemotePoolConfig(
                url=result.pool_server_url,
                worker_name="execution_manager",
                num_processes=1,
                threads_per_process=1,
            ),
        ),
    },
    graph=GraphConfig(
        nodes=[
            NodeConfig(
                factory="netrun.node_factories.from_function",
                factory_args={"func": "nodes.find_primes"},
                execution_config=NodeExecutionConfig(pools=["remote"]),
            ),
        ],
    ),
)

async with Net(client_config) as net:
    net.inject_data("find_primes", "start", [0])
    net.inject_data("find_primes", "stop", [1000])

    await net.run_until_blocked()

    results = net.flush_all_output_queues()
    primes = [v for vals in results.values() for v in vals[0]]
    primes.sort()

    print(f"Found {len(primes)} primes in [0, 1000)")
    print(f"First 10: {primes[:10]}")
    print(f"Last 10:  {primes[-10:]}")

    await net.request_pool_shutdown("remote")

ChannelClosed: Channel was shut down

## Logs

In [ ]:
net.print_all_logs()

## Cleanup

Delete the Hetzner Cloud server. **Always run this cell** to avoid ongoing charges.

In [ ]:
result.close_tunnel()
delete_server(SERVER_NAME)